This notebook tunes hyperparameters on top of the baseline results from `07_modeling_pipeline.ipynb`.

**Scope of This Notebook**

- **LightGBM is tuned on all 6 Sets** using RandomizedSearchCV **with the same TimeSeriesSplit** — it's the primary target and the model `09_shap_analysis.ipynb` will use consistently for cross-Set comparison. Under the chronological split, LightGBM was the default-hyperparameter winner in **4 of 6 Sets (C, D, E1, E2)**, while ranked #2 narrowly behind LogisticRegression in **both Set A and Set B**. LightGBM is retained across all 6 Sets rather than switching models per Set: these margins are well within noise (see `07`'s Wilcoxon results), and using one consistent model is necessary for a fair Set comparison and for consistent SHAP interpretation in `09_shap_analysis.ipynb` — a per-Set best of selection would confound differences in feature engineering methodology with differences in model type.

- **Each Set's strongest non-LightGBM model** gets a **lighter** tuning pass as a sanity check that LightGBM stays competitive once its best competitor is also tuned — not a full search. If this showed LightGBM being decisively beaten after tuning, it would undermine the choice above. Hence, a sanity check will be done to confirm LightGBM still wins after both sides are tuned.

- **Set x Model matrix comparison**: this matrix comparison will be redone with the tuned models for the final, reportable comparison. This is followed by a paired significance test (same Wilcoxon approach as `07`) to verify that the Set ranking still holds consistently across CV folds once LightGBM is tuned — the tuned matrix on its own only shows the ranking, not whether it's consistent enough to trust.

- **Threshold optimization for the final adopted model**: once LightGBM's hyperparameters are finalized, a calibrated decision threshold is selected per Set using train-side CV validation folds only (no test set used), replacing the default threshold=0.5 diagnosed as a source of overprediction in `07_modeling_pipeline.ipynb`.

## **0. Setup (shared pipeline imported from `modelling_utils.py`)**

In [1]:
# If not already installed in this environment:
# !pip install xgboost lightgbm --break-system-packages -q

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import os
from scipy.stats import wilcoxon

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score)

from modelling_utils import (
    RANDOM_STATE, TARGET, CAT_COLS, ID_COL, NON_FEATURE_COLS, RATING_COLS,
    SET_SCHEMA, GroupMedianImputer, build_pipeline, load_sets_and_split,
)

DATA_DIR = "../1_data/02_final_sets/"
METRICS_DIR = "../3_results/01_metrics/"  
MODEL_DIR = "fitted_pipelines/"       
os.makedirs(MODEL_DIR, exist_ok=True)

STAGE = "tuned"  # mirrors STAGE="baseline" naming convention from 07_modelling_pipeline file

In [2]:
SET_FILES = {
    "A": "01_set_a.csv", "B": "02_set_b.csv", "C": "03_set_c.csv",
    "D": "04_set_d.csv", "E1": "05_set_e1.csv", "E2": "06_set_e2.csv",
}
sets, splits = load_sets_and_split(DATA_DIR, SET_FILES)
print(f"Train size: {len(splits['A']['train'])}  |  Test size: {len(splits['A']['test'])}")

Train size: 18384  |  Test size: 4596


## **1. Load Baseline Results**

LightGBM is tuned as a baseline. The runner-up tuned as a sanity check is defined as the best non-LightGBM model for that Set. This stays meaningful even in a Set where LightGBM wasn't actually #1 at default hyperparameters. This is worth checking explicitly, since it can differ from the random-split baseline this scope was originally based on.

In [3]:
baseline = pd.read_csv(f"{METRICS_DIR}baseline_results.csv")

runner_up = {}
for set_name, grp in baseline.groupby("Set"):
    grp_ranked = grp.sort_values("Test_ROC_AUC", ascending=False).reset_index(drop=True)
    lgbm_rank = int(grp_ranked.index[grp_ranked["Model"] == "LightGBM"][0]) + 1
    if lgbm_rank != 1:
        print(f"Note: Set {set_name} -- LightGBM ranked #{lgbm_rank} at default hyperparameters.")
    best_alt = grp_ranked[grp_ranked["Model"] != "LightGBM"].iloc[0]
    runner_up[set_name] = best_alt["Model"]

print("\nRunner-Up per Set")
for k, v in runner_up.items():
    print(f"  {k}: {v}")

Note: Set A -- LightGBM ranked #2 at default hyperparameters.
Note: Set B -- LightGBM ranked #2 at default hyperparameters.

Runner-Up per Set
  A: LogisticRegression
  B: LogisticRegression
  C: XGBoost
  D: LogisticRegression
  E1: XGBoost
  E2: XGBoost


## **2. Hyperparameter Search Spaces**

In [4]:
LGBM_PARAM_DIST = {
    "model__n_estimators": [100, 200, 300, 500, 800],
    "model__num_leaves": [15, 31, 63, 127],
    "model__max_depth": [-1, 4, 6, 8, 10],
    "model__learning_rate": [0.01, 0.02, 0.05, 0.1, 0.2],
    "model__min_child_samples": [5, 10, 20, 50, 100],
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.01, 0.1, 1, 5],
    "model__reg_lambda": [0, 0.01, 0.1, 1, 5],
}

RF_PARAM_DIST = {
    "model__n_estimators": [200, 400, 600, 800],
    "model__max_depth": [None, 6, 10, 15, 20, 30],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5, 0.8],
}

XGB_PARAM_DIST = {
    "model__n_estimators": [100, 200, 300, 500, 800],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.02, 0.05, 0.1, 0.2],
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.01, 0.1, 1, 5],
    "model__reg_lambda": [0.1, 1, 5, 10],
    "model__min_child_weight": [1, 3, 5, 10],
}

LR_PARAM_DIST = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__solver": ["lbfgs", "liblinear"],
}

PARAM_DIST_BY_MODEL = {
    "LightGBM": LGBM_PARAM_DIST,
    "RandomForest": RF_PARAM_DIST,
    "XGBoost": XGB_PARAM_DIST,
    "LogisticRegression": LR_PARAM_DIST,
}

## **3. Tune LightGBM on All 6 Sets**

`n_iter=30` random hyperparameter draws per Set, evaluated with TimeSeriesSplit(5), selecting on `roc_auc`. PR-AUC per fold is also recorded (for comparability with baseline_results.csv), but selection/refit is driven by ROC-AUC only, consistent with `07_modelling_pipeline.ipynb`'s conclusion that ROC-AUC is the primary metric.

In [5]:
N_ITER_MAIN = 30
tscv = TimeSeriesSplit(n_splits=5)

tuned_lgbm = {}              # set_name -> fitted best_estimator_ (full Pipeline)
tuned_lgbm_fold_scores = {}  # set_name -> array of 5 per-fold roc_auc scores (best config)
tuned_lgbm_params = {}


SCORING = {"roc_auc": "roc_auc", "average_precision": "average_precision"}

for set_name in SET_FILES:
    y_train = splits[set_name]["train"][TARGET]
    X_train = splits[set_name]["train"].drop(columns=[TARGET])
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    base_pipe = build_pipeline(set_name, "LightGBM", X_train, scale_pos_weight, n_jobs=1)
    search = RandomizedSearchCV(
        base_pipe, LGBM_PARAM_DIST, n_iter=N_ITER_MAIN, cv=tscv,
        scoring=SCORING, refit="roc_auc", random_state=RANDOM_STATE, n_jobs=-1,
    )
    search.fit(X_train, y_train)

    tuned_lgbm[set_name] = search.best_estimator_
    tuned_lgbm_params[set_name] = search.best_params_
    fold_cols = [f"split{i}_test_roc_auc" for i in range(tscv.get_n_splits())]
    tuned_lgbm_fold_scores[set_name] = np.array(
        [search.cv_results_[c][search.best_index_] for c in fold_cols]
    )

    print(f"Set {set_name}: best CV ROC-AUC = {search.best_score_:.4f}")
    print(f"  best params: {search.best_params_}")

Set A: best CV ROC-AUC = 0.9844
  best params: {'model__subsample': 0.9, 'model__reg_lambda': 0, 'model__reg_alpha': 1, 'model__num_leaves': 31, 'model__n_estimators': 100, 'model__min_child_samples': 5, 'model__max_depth': 10, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.6}
Set B: best CV ROC-AUC = 0.9463
  best params: {'model__subsample': 0.8, 'model__reg_lambda': 5, 'model__reg_alpha': 5, 'model__num_leaves': 63, 'model__n_estimators': 500, 'model__min_child_samples': 50, 'model__max_depth': 6, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1.0}
Set C: best CV ROC-AUC = 0.9377
  best params: {'model__subsample': 0.6, 'model__reg_lambda': 0.01, 'model__reg_alpha': 0.01, 'model__num_leaves': 31, 'model__n_estimators': 300, 'model__min_child_samples': 50, 'model__max_depth': 8, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.8}
Set D: best CV ROC-AUC = 0.9529
  best params: {'model__subsample': 1.0, 'model__reg_lambda': 1, 'model__reg_alpha': 0.1, 'm

## **4. Lightly Tune Each Set's Runner-Up Model**

Smaller search budget (`n_iter=10`). The goal here is only to confirm LightGBM's advantage isn't an artifact of LightGBM having been implicitly favored by default hyperparameters while its competitor was left untuned.

In [6]:
N_ITER_LIGHT = 10

tuned_runner_up = {}
tuned_runner_up_fold_scores = {}
tuned_runner_up_params = {}

for set_name, model_name in runner_up.items():
    y_train = splits[set_name]["train"][TARGET]
    X_train = splits[set_name]["train"].drop(columns=[TARGET])
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    base_pipe = build_pipeline(set_name, model_name, X_train, scale_pos_weight, n_jobs=1)
    param_dist = PARAM_DIST_BY_MODEL[model_name]
    search = RandomizedSearchCV(
        base_pipe, param_dist, n_iter=N_ITER_LIGHT, cv=tscv,
        scoring=SCORING, refit="roc_auc", random_state=RANDOM_STATE, n_jobs=-1,
    )
    search.fit(X_train, y_train)

    tuned_runner_up[set_name] = search.best_estimator_
    tuned_runner_up_params[set_name] = search.best_params_
    fold_cols = [f"split{i}_test_roc_auc" for i in range(tscv.get_n_splits())]
    tuned_runner_up_fold_scores[set_name] = np.array(
        [search.cv_results_[c][search.best_index_] for c in fold_cols]
    )

    print(f"Set {set_name} ({model_name}): best CV ROC-AUC = {search.best_score_:.4f}")

Set A (LogisticRegression): best CV ROC-AUC = 0.9836
Set B (LogisticRegression): best CV ROC-AUC = 0.9445
Set C (XGBoost): best CV ROC-AUC = 0.9381
Set D (LogisticRegression): best CV ROC-AUC = 0.9510
Set E1 (XGBoost): best CV ROC-AUC = 0.9714
Set E2 (XGBoost): best CV ROC-AUC = 0.9703


## **5. Evaluate Models: Forward-Chaining CV (train) + Held-out Test**

In [7]:
tuned_results = []

for set_name in SET_FILES:
    y_train = splits[set_name]["train"][TARGET]
    y_test = splits[set_name]["test"][TARGET]
    X_test = splits[set_name]["test"].drop(columns=[TARGET])

    train_positive_rate = y_train.mean()
    test_positive_rate = y_test.mean()

    for label, model_dict, cv_scores_dict in [
        ("LightGBM", tuned_lgbm, tuned_lgbm_fold_scores),
        (runner_up[set_name], tuned_runner_up, tuned_runner_up_fold_scores),
    ]:
        pipe = model_dict[set_name]
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]

        tuned_results.append({
            "Set": set_name, "Model": label,
            "CV_ROC_AUC_mean": cv_scores_dict[set_name].mean(),
            "CV_ROC_AUC_std": cv_scores_dict[set_name].std(),
            "Test_Accuracy": accuracy_score(y_test, y_pred),
            "Test_Precision": precision_score(y_test, y_pred),
            "Test_Recall": recall_score(y_test, y_pred),
            "Test_F1": f1_score(y_test, y_pred),
            "Test_ROC_AUC": roc_auc_score(y_test, y_proba),
            "Test_PR_AUC": average_precision_score(y_test, y_proba),
            "Test_PR_AUC_baseline": test_positive_rate,  # random-classifier PR-AUC for this split
            "Train_positive_rate": train_positive_rate,
            "Test_positive_rate": test_positive_rate,
            "Predicted_positive_rate": y_pred.mean(),  # threshold=0.5 diagnostic
        })

tuned_results_df = pd.DataFrame(tuned_results).sort_values(["Set", "Test_ROC_AUC"], ascending=[True, False])
pd.set_option("display.width", 160)
print(tuned_results_df.round(4).to_string(index=False))

Set              Model  CV_ROC_AUC_mean  CV_ROC_AUC_std  Test_Accuracy  Test_Precision  Test_Recall  Test_F1  Test_ROC_AUC  Test_PR_AUC  Test_PR_AUC_baseline  Train_positive_rate  Test_positive_rate  Predicted_positive_rate
  A           LightGBM           0.9844          0.0052         0.9532          0.8700       0.9417   0.9044        0.9862       0.9670                 0.235               0.3593               0.235                   0.2544
  A LogisticRegression           0.9836          0.0056         0.9530          0.8649       0.9481   0.9046        0.9860       0.9664                 0.235               0.3593               0.235                   0.2576
  B           LightGBM           0.9463          0.0054         0.9027          0.7644       0.8472   0.8037        0.9476       0.8917                 0.235               0.3593               0.235                   0.2604
  B LogisticRegression           0.9445          0.0055         0.8856          0.7046       0.8833   0.

## **6. Sanity check: Does LightGBM still win after both sides are tuned?**

In [8]:
# Safeguard: runner_up must match what tuned_results_df actually contains
for set_name in SET_FILES:
    available_models = tuned_results_df[tuned_results_df["Set"] == set_name]["Model"].tolist()
    assert runner_up[set_name] in available_models, (
        f"Set {set_name}: runner_up says '{runner_up[set_name]}', but tuned_results_df "
        f"only has {available_models}. Likely stale state -- re-run notebook in order "
        f"from Section 2 (Load Baseline Results)."
    )

lgbm_still_wins = []
for set_name in SET_FILES:
    row = tuned_results_df[tuned_results_df["Set"] == set_name].sort_values("Test_ROC_AUC", ascending=False)
    winner = row.iloc[0]["Model"]
    lgbm_still_wins.append(winner == "LightGBM")
    print(f"Set {set_name}: winner after tuning = {winner} "
          f"(LightGBM={row[row['Model']=='LightGBM']['Test_ROC_AUC'].values[0]:.4f}, "
          f"{runner_up[set_name]}={row[row['Model']==runner_up[set_name]]['Test_ROC_AUC'].values[0]:.4f})")

print(f"\nLightGBM wins in {sum(lgbm_still_wins)}/6 Sets after tuning.")

Set A: winner after tuning = LightGBM (LightGBM=0.9862, LogisticRegression=0.9860)
Set B: winner after tuning = LightGBM (LightGBM=0.9476, LogisticRegression=0.9457)
Set C: winner after tuning = XGBoost (LightGBM=0.9414, XGBoost=0.9420)
Set D: winner after tuning = LightGBM (LightGBM=0.9539, LogisticRegression=0.9509)
Set E1: winner after tuning = XGBoost (LightGBM=0.9685, XGBoost=0.9692)
Set E2: winner after tuning = LightGBM (LightGBM=0.9672, XGBoost=0.9670)

LightGBM wins in 4/6 Sets after tuning.


> ### **Sanity Check Result**
>
> After both LightGBM and each Set's strongest competitor were tuned, LightGBM wins in 4 of 6 Sets (A, B, D, E2), with the runner-up model winning narrowly in the remaining 2 (C, E1). All margins are within 0.0002-0.0030 ROC-AUC which is acceptable.
>
> This confirms LightGBM's advantage is not an artifact of the competitor being left untuned. Given this near-parity, **LightGBM is retained as the model for all six Sets**, prioritizing a consistent, fair Set comparison and consistent SHAP interpretation in `09_shap_analysis.ipynb` over a per-Set "best of" selection that would offer no meaningful performance gain.

## **7. Final Set x Model Matrix**

In [9]:
lgbm_matrix = tuned_results_df[tuned_results_df["Model"] == "LightGBM"].set_index("Set")
print("=== Tuned LightGBM — Test ROC-AUC by Set ===")
print(lgbm_matrix[["Test_ROC_AUC", "Test_PR_AUC", "Test_PR_AUC_baseline", "Test_F1"]]
      .round(4).sort_values("Test_ROC_AUC", ascending=False))

set_order = lgbm_matrix.sort_values("Test_ROC_AUC", ascending=False).index.tolist()
print(f"\nFinal Set ranking (tuned LightGBM): {' > '.join(set_order)}")

=== Tuned LightGBM — Test ROC-AUC by Set ===
     Test_ROC_AUC  Test_PR_AUC  Test_PR_AUC_baseline  Test_F1
Set                                                          
A          0.9862       0.9670                 0.235   0.9044
E1         0.9685       0.9446                 0.235   0.9075
E2         0.9672       0.9428                 0.235   0.8870
D          0.9539       0.9060                 0.235   0.8238
B          0.9476       0.8917                 0.235   0.8037
C          0.9414       0.8720                 0.235   0.7857

Final Set ranking (tuned LightGBM): A > E1 > E2 > D > B > C


## **8. Paired Significance Test**

In [10]:
# TimeSeriesSplit(n_splits=5) gives only 5 paired observations per comparison.
# Wilcoxon signed-rank test has low power at this sample size, so p-values here should
# be read as directional/suggestive, not as confirmatory evidence of significance.

tuned_paired_results = []

def paired_compare(set1, set2, label):
    a = tuned_lgbm_fold_scores[set1]
    b = tuned_lgbm_fold_scores[set2]
    diff = a - b
    try:
        stat, p = wilcoxon(a, b)
    except ValueError:
        # all paired differences are exactly zero
        p = float("nan")
    winner = set1 if diff.mean() > 0 else set2
    sig = "significant" if p < 0.05 else "not significant"
    print(f"{label}: {set1}={a.mean():.4f}  {set2}={b.mean():.4f}  "
          f"mean_diff={diff.mean():+.4f}  wilcoxon p={p:.4f}  -> {winner} higher ({sig}, n=5 folds)")
    tuned_paired_results.append({
        "Comparison": label, "Set1": set1, "Set2": set2, "Model": "LightGBM (tuned)",
        "Set1_mean_ROC_AUC": a.mean(), "Set2_mean_ROC_AUC": b.mean(),
        "Mean_diff": diff.mean(), "Wilcoxon_p": p, "Winner": winner,
        "Significant_at_0.05": p < 0.05,
    })

# --- Numeric baseline vs text-only methods (does text alone approach numeric ratings?) ---
paired_compare("A", "E1", "Numeric vs BERT (full inference)")
paired_compare("A", "E2", "Numeric vs BERT (keyword-gated)")

# --- Rule-based / lexicon methods vs each other and vs BERT ---
paired_compare("B", "C", "VADER (document) vs Rule-based (aspect)")
paired_compare("B", "D", "VADER (document) vs Rule-based (aspect, variant D)")
paired_compare("C", "D", "Rule-based (aspect) vs Rule-based (aspect, variant D)")
paired_compare("C", "E1", "Rule-based vs BERT (full inference)")
paired_compare("C", "E2", "Rule-based vs BERT (keyword-gated)")
paired_compare("D", "E1", "Rule-based (aspect, variant D) vs BERT (full inference)")
paired_compare("B", "E1", "VADER (document) vs BERT (full inference)")

tuned_paired_df = pd.DataFrame(tuned_paired_results)
print(f"\nTotal comparisons: {len(tuned_paired_df)}")

Numeric vs BERT (full inference): A=0.9844  E1=0.9716  mean_diff=+0.0128  wilcoxon p=0.0625  -> A higher (not significant, n=5 folds)
Numeric vs BERT (keyword-gated): A=0.9844  E2=0.9699  mean_diff=+0.0145  wilcoxon p=0.0625  -> A higher (not significant, n=5 folds)
VADER (document) vs Rule-based (aspect): B=0.9463  C=0.9377  mean_diff=+0.0086  wilcoxon p=0.0625  -> B higher (not significant, n=5 folds)
VADER (document) vs Rule-based (aspect, variant D): B=0.9463  D=0.9529  mean_diff=-0.0066  wilcoxon p=0.0625  -> D higher (not significant, n=5 folds)
Rule-based (aspect) vs Rule-based (aspect, variant D): C=0.9377  D=0.9529  mean_diff=-0.0152  wilcoxon p=0.0625  -> D higher (not significant, n=5 folds)
Rule-based vs BERT (full inference): C=0.9377  E1=0.9716  mean_diff=-0.0340  wilcoxon p=0.0625  -> E1 higher (not significant, n=5 folds)
Rule-based vs BERT (keyword-gated): C=0.9377  E2=0.9699  mean_diff=-0.0322  wilcoxon p=0.0625  -> E2 higher (not significant, n=5 folds)
Rule-based (a

**Test ROC-AUC by Set: Baseline (Best Model, Default Hyperparameters) vs. Final Model (Tuned LightGBM)**

| Set | Baseline Best Model | Baseline Test_ROC_AUC | Tuned Test_ROC_AUC | ΔROC_AUC | Baseline Test_F1 | Tuned Test_F1 | ΔF1 |
|---|---|---|---|---|---|---|---|
| A | LogisticRegression | 0.9861 | 0.9862 | +0.0001 | 0.9064 | 0.9044 | -0.0020 |
| E1 | LightGBM | 0.9681 | 0.9685 | +0.0004 | 0.9041 | 0.9075 | +0.0034 |
| E2 | LightGBM | 0.9654 | 0.9672 | +0.0018 | 0.8844 | 0.8870 | +0.0026 |
| D | LightGBM | 0.9516 | 0.9539 | +0.0023 | 0.8248 | 0.8238 | -0.0010 |
| B | LogisticRegression | 0.9455 | 0.9476 | +0.0021 | 0.7971 | 0.8037 | +0.0066 |
| C | LightGBM | 0.9410 | 0.9414 | +0.0004 | 0.7792 | 0.7857 | +0.0065 |
>
>
**Test ROC-AUC by Set (Final Model: Tuned LightGBM)**

| Set | Test_ROC_AUC | Test_PR_AUC | Test_PR_AUC_baseline | Test_F1 |
|---|---|---|---|---|
| A | 0.9862 | 0.9670 | 0.235 | 0.9044 |
| E1 | 0.9685 | 0.9446 | 0.235 | 0.9075 |
| E2 | 0.9672 | 0.9428 | 0.235 | 0.8870 |
| D | 0.9539 | 0.9060 | 0.235 | 0.8238 |
| B | 0.9476 | 0.8917 | 0.235 | 0.8037 |
| C | 0.9414 | 0.8720 | 0.235 | 0.7857 |

**Final Set ranking: A > E1 > E2 > D > B > C**

> ### **Key Insight**
>
> - **Ranking unchanged after tuning:** The Set ranking (A > E1 > E2 > D > B > C)
  established with default hyperparameters in `07_modelling_pipeline.ipynb` holds
  **exactly the same** after LightGBM was tuned. This indicates the ranking reflects
  a fundamental difference in the information content of each Set, not an artifact of
  model choice or hyperparameter configuration.
>
> - **Tuning gains are consistently small (ΔROC_AUC within +0.0001 to +0.0023):**
  Baseline (default hyperparameters) was already close to optimal, and LightGBM
  appears relatively insensitive to hyperparameters on this dataset. Practically,
  this suggests that at this data scale, near-default performance is achievable
  without heavy tuning investment -- useful to know when weighing tuning cost against
  expected gain in future iterations.
>
> - **Note on Set A and D:** These are the only two Sets where switching from each
  Set's own best baseline model to the single tuned LightGBM caused a (very small)
  F1 decrease (-0.0020 and -0.0010 respectively), since their baseline-best model
  wasn't LightGBM to begin with (A: LogisticRegression) or LightGBM's F1 shifted
  slightly after tuning (D). ROC-AUC still improved for both, and the magnitude is
  well within noise -- consistent with the decision to standardize on LightGBM
  across all Sets rather than mix model types.
>
> - **All 9 pairwise comparisons agree in direction at p=0.0625:** The strongest
  possible consistency signal at n=5 folds (every fold agreed on the winner in every
  comparison). This covers all adjacent pairs (A-E1, E1-E2, E2-D, D-B, B-C) plus key
  cross-comparisons (A-E2, B-D, C-D), leaving no part of the ranking unverified.
>
> - **PR-AUC vs. baseline confirms practical value, not just relative ranking:** Every
  Set's `Test_PR_AUC` (0.87-0.97) is far above `Test_PR_AUC_baseline` (0.235), so even
  the weakest Set (C) remains a substantial, useful predictor.
>
> - **Practical takeaways:**
>   - Set A (numeric ratings) is the strongest predictor, but requires users to supply star ratings, not just free text.
>   - Set E1 (BERT, full inference) is the best text-only method, within ~0.018 ROC-AUC of Set A.
>   - BERT-based methods (E1, E2) consistently outperform rule-based/lexicon methods (B, C, D), with Set D as the strongest non-BERT text method.
>
> - **Caveat on significance:** With only 5 CV folds, no single Wilcoxon test can formally clear p<0.05 -- a structural limit of sample size, not weak evidence. The ranking's credibility instead rests on convergence of independent checks: consistent across all 4 model families (baseline), consistent across CV folds, and now confirmed unchanged after hyperparameter tuning.

## **9. Threshold Calibration (for final adopted model)**

In [11]:
from sklearn.metrics import confusion_matrix

gap_check_tuned = tuned_results_df[
    tuned_results_df["Model"] == "LightGBM"
][["Set", "Model", "Test_positive_rate", "Predicted_positive_rate"]].copy()

gap_check_tuned["Overprediction_gap"] = (
    gap_check_tuned["Predicted_positive_rate"] - gap_check_tuned["Test_positive_rate"]
)

print("=== Predicted vs Actual positive rate (threshold=0.5, tuned LightGBM) ===")
print(gap_check_tuned.sort_values("Overprediction_gap", ascending=False).to_string(index=False))


top_gap_tuned = gap_check_tuned.sort_values("Overprediction_gap", ascending=False).head(3)

print("=== Confusion matrices for top overprediction-gap Sets (tuned LightGBM, threshold=0.5) ===")
for _, row in top_gap_tuned.iterrows():
    set_name = row["Set"]
    pipe = tuned_lgbm[set_name]
    X_test = splits[set_name]["test"].drop(columns=[TARGET])
    y_test = splits[set_name]["test"][TARGET]
    y_pred = pipe.predict(X_test)  # default threshold=0.5

    cm = confusion_matrix(y_test, y_pred)
    print(f"\nSet {set_name}  (gap={row['Overprediction_gap']:+.4f})")
    display(pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))

=== Predicted vs Actual positive rate (threshold=0.5, tuned LightGBM) ===
Set    Model  Test_positive_rate  Predicted_positive_rate  Overprediction_gap
  C LightGBM            0.234987                 0.270670            0.035683
  D LightGBM            0.234987                 0.265231            0.030244
  B LightGBM            0.234987                 0.260444            0.025457
  A LightGBM            0.234987                 0.254352            0.019365
 E2 LightGBM            0.234987                 0.229112           -0.005875
 E1 LightGBM            0.234987                 0.228242           -0.006745
=== Confusion matrices for top overprediction-gap Sets (tuned LightGBM, threshold=0.5) ===

Set C  (gap=+0.0357)


,Pred 0,Pred 1
Actual 0,3185,331
Actual 1,167,913



Set D  (gap=+0.0302)


,Pred 0,Pred 1
Actual 0,3244,272
Actual 1,133,947



Set B  (gap=+0.0255)


,Pred 0,Pred 1
Actual 0,3234,282
Actual 1,165,915


In [12]:
def find_calibrated_threshold(set_name, best_params, X_train, y_train, tscv,
                               threshold_grid=np.arange(0.05, 0.95, 0.005)):
    """
    Finds a decision threshold using only train-side CV validation folds.
    For each fold, refits with the already-tuned hyperparameters, then finds the
    threshold whose predicted positive rate on that fold's validation split most
    closely matches the validation split's own true positive rate.
    Returns the mean threshold across folds (and the per-fold thresholds, for
    transparency about how stable the estimate is).
    """
    fold_thresholds = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
        fold_pipe = build_pipeline(set_name, "LightGBM", X_tr, scale_pos_weight, n_jobs=1)
        fold_pipe.set_params(**best_params)
        fold_pipe.fit(X_tr, y_tr)

        val_proba = fold_pipe.predict_proba(X_val)[:, 1]
        target_rate = y_val.mean()

        best_t, best_diff = 0.5, float("inf")
        for t in threshold_grid:
            diff = abs((val_proba >= t).mean() - target_rate)
            if diff < best_diff:
                best_t, best_diff = t, diff
        fold_thresholds.append(best_t)

    return np.mean(fold_thresholds), fold_thresholds


calibrated_thresholds = {}
threshold_diagnostics = []

for set_name in SET_FILES:
    y_train = splits[set_name]["train"][TARGET]
    X_train = splits[set_name]["train"].drop(columns=[TARGET])

    mean_t, fold_ts = find_calibrated_threshold(
        set_name, tuned_lgbm_params[set_name], X_train, y_train, tscv
    )
    calibrated_thresholds[set_name] = round(mean_t, 3)

    threshold_diagnostics.append({
        "Set": set_name,
        "Calibrated_threshold": round(mean_t, 3),
        "Threshold_std": round(np.std(fold_ts), 3),
        "Fold_thresholds": [round(t, 3) for t in fold_ts],
    })

    print(f"{set_name}: calibrated threshold = {mean_t:.3f}  (fold thresholds: {[round(t,3) for t in fold_ts]})")

A: calibrated threshold = 0.605  (fold thresholds: [np.float64(0.565), np.float64(0.6), np.float64(0.59), np.float64(0.625), np.float64(0.645)])
B: calibrated threshold = 0.594  (fold thresholds: [np.float64(0.65), np.float64(0.585), np.float64(0.55), np.float64(0.59), np.float64(0.595)])
C: calibrated threshold = 0.601  (fold thresholds: [np.float64(0.685), np.float64(0.6), np.float64(0.575), np.float64(0.545), np.float64(0.6)])
D: calibrated threshold = 0.598  (fold thresholds: [np.float64(0.665), np.float64(0.59), np.float64(0.57), np.float64(0.56), np.float64(0.605)])
E1: calibrated threshold = 0.417  (fold thresholds: [np.float64(0.51), np.float64(0.385), np.float64(0.37), np.float64(0.4), np.float64(0.42)])
E2: calibrated threshold = 0.437  (fold thresholds: [np.float64(0.56), np.float64(0.45), np.float64(0.43), np.float64(0.365), np.float64(0.38)])


In [13]:
calibrated_results = []

for set_name in SET_FILES:
    y_test = splits[set_name]["test"][TARGET]
    X_test = splits[set_name]["test"].drop(columns=[TARGET])
    pipe = tuned_lgbm[set_name]
    y_proba = pipe.predict_proba(X_test)[:, 1]

    t = calibrated_thresholds[set_name]
    y_pred_calibrated = (y_proba >= t).astype(int)

    # "before" row, already computed in Section 6/tuned_results_df, for comparison
    before = tuned_results_df[(tuned_results_df["Set"] == set_name) &
                               (tuned_results_df["Model"] == "LightGBM")].iloc[0]

    calibrated_results.append({
        "Set": set_name,
        "Threshold_used": t,
        "Test_positive_rate": y_test.mean(),
        "Predicted_rate_before (t=0.5)": before["Predicted_positive_rate"],
        "Predicted_rate_after": y_pred_calibrated.mean(),
        "F1_before": before["Test_F1"],
        "F1_after": f1_score(y_test, y_pred_calibrated),
        "Accuracy_before": before["Test_Accuracy"],
        "Accuracy_after": accuracy_score(y_test, y_pred_calibrated),
    })

calibrated_df = pd.DataFrame(calibrated_results)
calibrated_df["Gap_before"] = calibrated_df["Predicted_rate_before (t=0.5)"] - calibrated_df["Test_positive_rate"]
calibrated_df["Gap_after"] = calibrated_df["Predicted_rate_after"] - calibrated_df["Test_positive_rate"]

pd.set_option("display.width", 160)
print("=== Threshold Calibration: Before (t=0.5) vs After ===")
print(calibrated_df.round(4).to_string(index=False))

=== Threshold Calibration: Before (t=0.5) vs After ===
Set  Threshold_used  Test_positive_rate  Predicted_rate_before (t=0.5)  Predicted_rate_after  F1_before  F1_after  Accuracy_before  Accuracy_after  Gap_before  Gap_after
  A           0.605               0.235                         0.2544                0.2454     0.9044    0.9130           0.9532          0.9582      0.0194     0.0104
  B           0.594               0.235                         0.2604                0.2419     0.8037    0.8175           0.9027          0.9130      0.0255     0.0070
  C           0.601               0.235                         0.2707                0.2391     0.7857    0.7967           0.8916          0.9036      0.0357     0.0041
  D           0.598               0.235                         0.2652                0.2472     0.8238    0.8294           0.9119          0.9178      0.0302     0.0122
 E1           0.417               0.235                         0.2282                0.2378   

> ### **Key Insight**
>
| Set | Threshold | Interpretation |
|---|---|---|
| A | 0.605 | Model over-predicts positives by default → threshold raised |
| B | 0.594 | Same direction as A |
| C | 0.601 | Same direction; largest fold-to-fold variance (0.545-0.685), consistent with C being the weakest-performing Set overall |
| D | 0.598 | Same direction as A/B/C |
| E1 | 0.417 | Model under-predicts positives by default → threshold lowered |
| E2 | 0.437 | Same direction as E1 |
>
> Calibration corrects the direction of miscalibration specific to each Set's model — raising the bar for Sets that over-predict (A, B, C, D) and lowering it for Sets that under-predict (E1, E2), while keeping F1 balanced (improved in 4/6 Sets, marginal <0.01 decrease in the other 2). These are the final thresholds to pair with each Set's tuned LightGBM pipeline at deployment.

## **10. Save Results**

In [15]:
prefix = STAGE 

lgbm_matrix.to_csv(f"{METRICS_DIR}{prefix}_lgbm_roc_auc_by_set.csv")
print(f"Saved: {METRICS_DIR}{prefix}_lgbm_roc_auc_by_set.csv")

import json

for set_name in SET_FILES:
    threshold_path = f"{MODEL_DIR}lgbm_tuned_{set_name}_threshold.json"
    with open(threshold_path, "w") as f:
        json.dump({
            "set": set_name,
            "model": "LightGBM",
            "threshold": calibrated_thresholds[set_name],
            "calibrated_on": "train-side TimeSeriesSplit CV validation folds only",
            "note": "Use pipe.predict_proba(X)[:, 1] >= threshold at inference time, "
                     "NOT pipe.predict(X) (which hard-codes 0.5).",
        }, f, indent=2)
    print(f"Saved: {threshold_path}")

calibrated_df.to_csv(f"{METRICS_DIR}{prefix}_calibrated_thresholds.csv", index=False)
print(f"Saved: {METRICS_DIR}{prefix}_calibrated_thresholds.csv")

Saved: ../3_results/01_metrics/tuned_lgbm_roc_auc_by_set.csv
Saved: fitted_pipelines/lgbm_tuned_A_threshold.json
Saved: fitted_pipelines/lgbm_tuned_B_threshold.json
Saved: fitted_pipelines/lgbm_tuned_C_threshold.json
Saved: fitted_pipelines/lgbm_tuned_D_threshold.json
Saved: fitted_pipelines/lgbm_tuned_E1_threshold.json
Saved: fitted_pipelines/lgbm_tuned_E2_threshold.json
Saved: ../3_results/01_metrics/tuned_calibrated_thresholds.csv


In [16]:
import os
print(sorted(os.listdir("fitted_pipelines")))

['A_LightGBM.joblib', 'A_LogisticRegression.joblib', 'A_RandomForest.joblib', 'A_XGBoost.joblib', 'B_LightGBM.joblib', 'B_LogisticRegression.joblib', 'B_RandomForest.joblib', 'B_XGBoost.joblib', 'C_LightGBM.joblib', 'C_LogisticRegression.joblib', 'C_RandomForest.joblib', 'C_XGBoost.joblib', 'D_LightGBM.joblib', 'D_LogisticRegression.joblib', 'D_RandomForest.joblib', 'D_XGBoost.joblib', 'E1_LightGBM.joblib', 'E1_LogisticRegression.joblib', 'E1_RandomForest.joblib', 'E1_XGBoost.joblib', 'E2_LightGBM.joblib', 'E2_LogisticRegression.joblib', 'E2_RandomForest.joblib', 'E2_XGBoost.joblib', 'lgbm_tuned_A.joblib', 'lgbm_tuned_A_threshold.json', 'lgbm_tuned_B.joblib', 'lgbm_tuned_B_threshold.json', 'lgbm_tuned_C.joblib', 'lgbm_tuned_C_threshold.json', 'lgbm_tuned_D.joblib', 'lgbm_tuned_D_threshold.json', 'lgbm_tuned_E1.joblib', 'lgbm_tuned_E1_threshold.json', 'lgbm_tuned_E2.joblib', 'lgbm_tuned_E2_threshold.json', 'logisticregression_tuned_A.joblib', 'logisticregression_tuned_B.joblib', 'logist